In [35]:
# All Imports
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
from datetime import datetime
from collections import Counter

# Scikit-learn imports
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Machine learning models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

# XGBoost (Transformer-based)
from xgboost import XGBClassifier

# Hyperparameter optimization
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# Evaluation metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, matthews_corrcoef)

import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU Not Found")

# Visualization settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14

GPU: NVIDIA GeForce RTX 4060


In [36]:
from xgboost import XGBClassifier

model = XGBClassifier(
    tree_method="hist",
    device="cuda",
    random_state=42
)

In [8]:
# Feature Engineering and Preprocessing
# Load Datasets

df = pd.read_csv('./data/raw/finguard-fraud-detection-dataset.csv')
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [9]:
df.shape

(6362620, 11)

In [10]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [11]:
print("\n Class Distribution:")
print(f"  - Fraud (isFraud=1): {df['isFraud'].sum():,} ({df['isFraud'].sum()/len(df)*100:.4f}%)")
print(f"  - Non-Fraud (isFraud=0): {len(df)-df['isFraud'].sum():,} ({100-df['isFraud'].sum()/len(df)*100:.4f}%)")


 Class Distribution:
  - Fraud (isFraud=1): 8,213 (0.1291%)
  - Non-Fraud (isFraud=0): 6,354,407 (99.8709%)


In [14]:
# Remove sensitive columns
sensitive_cols = ['oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 
                  'newbalanceDest', 'nameOrig', 'nameDest']

df_clean = df.drop(sensitive_cols, axis=1)

print(f"✓ Dropped sensitive columns: {sensitive_cols}")
print(f"✓ New shape: {df_clean.shape}")
print(f"✓ Remaining columns: {df_clean.columns.tolist()}")

✓ Dropped sensitive columns: ['oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'nameOrig', 'nameDest']
✓ New shape: (6362620, 5)
✓ Remaining columns: ['step', 'type', 'amount', 'isFraud', 'isFlaggedFraud']


In [15]:
print(f"✓ Missing values after drop: {df_clean.isnull().sum().sum()}")

✓ Missing values after drop: 0


In [16]:
# FEATURE ENGINEERING - CATEGORICAL ENCODING
le = LabelEncoder()
df_clean['type_encoded'] = le.fit_transform(df_clean['type'])

In [18]:
# Create mapping dictionary
type_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"✓ Transaction type encoding:")
for trans_type, code in type_mapping.items():
    print(f"  - {trans_type}: {code}")

# Show distribution of transaction types
print(f"\n✓ Transaction Type Distribution:")
print(df_clean['type'].value_counts())


✓ Transaction type encoding:
  - CASH_IN: 0
  - CASH_OUT: 1
  - DEBIT: 2
  - PAYMENT: 3
  - TRANSFER: 4

✓ Transaction Type Distribution:
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64


In [19]:
# FEATURE ENGINEERING - TIME-BASED FEATURES
# Extract time components from step (1 step = 1 hour)
df_clean['hour_of_day'] = df_clean['step'] % 24
df_clean['day_of_week'] = (df_clean['step'] // 24) % 7
df_clean['week_of_month'] = (df_clean['step'] // 168) + 1  # 24*7=168 hours in a week

print(f"✓ Created time-based features:")
print(f"  - hour_of_day: {df_clean['hour_of_day'].min()} to {df_clean['hour_of_day'].max()}")
print(f"  - day_of_week: {df_clean['day_of_week'].min()} to {df_clean['day_of_week'].max()}")
print(f"  - week_of_month: {df_clean['week_of_month'].min()} to {df_clean['week_of_month'].max()}")

✓ Created time-based features:
  - hour_of_day: 0 to 23
  - day_of_week: 0 to 6
  - week_of_month: 1 to 5


In [22]:
# FEATURE ENGINEERING - AMOUNT TRANSFORMATIONS
# Log transformation
df_clean['log_amount'] = np.log1p(df_clean['amount'])

# Square root transformation (alternative)
df_clean['sqrt_amount'] = np.sqrt(df_clean['amount'])

# Standardized amount
mean_amount = df_clean['amount'].mean()
std_amount = df_clean['amount'].std()
df_clean['zscore_amount'] = (df_clean['amount'] - mean_amount) / std_amount

print(f"✓ Created amount transformations:")
print(f"  - log_amount: log(amount + 1)")
print(f"  - sqrt_amount: sqrt(amount)")
print(f"  - zscore_amount: standardized amount")

print(f"\n📋 Amount Statistics:")
print(df_clean['amount'].describe().round(2))

✓ Created amount transformations:
  - log_amount: log(amount + 1)
  - sqrt_amount: sqrt(amount)
  - zscore_amount: standardized amount

📋 Amount Statistics:
count     6362620.00
mean       179861.90
std        603858.23
min             0.00
25%         13389.57
50%         74871.94
75%        208721.48
max      92445516.64
Name: amount, dtype: float64


In [23]:
# High-value transaction flags (based on 200,000 threshold from dataset)
df_clean['is_high_value'] = (df_clean['amount'] > 200000).astype(int)
df_clean['is_very_high_value'] = (df_clean['amount'] > df_clean['amount'].quantile(0.99)).astype(int)

print(f"✓ Created amount flags:")
print(f"  - is_high_value: amount > 200,000 (dataset threshold)")
print(f"  - is_very_high_value: amount > 99th percentile")


✓ Created amount flags:
  - is_high_value: amount > 200,000 (dataset threshold)
  - is_very_high_value: amount > 99th percentile


In [25]:
# EATURE ENGINEERING - FRAUD PATTERN DETECTION

# Create transaction pair features
# For each transaction, check if there's a matching transaction with same amount
df_clean['amount_count'] = df_clean.groupby('amount')['step'].transform('count')
df_clean['is_duplicate_amount'] = (df_clean['amount_count'] > 1).astype(int)

# Transaction type combination features
df_clean['is_cash_out'] = (df_clean['type'] == 'CASH_OUT').astype(int)
df_clean['is_transfer'] = (df_clean['type'] == 'TRANSFER').astype(int)
df_clean['is_payment'] = (df_clean['type'] == 'PAYMENT').astype(int)
df_clean['is_cash_in'] = (df_clean['type'] == 'CASH_IN').astype(int)

print(f"✓ Created fraud pattern features:")
print(f"  - amount_count: number of transactions with same amount")
print(f"  - is_duplicate_amount: flag for duplicate amounts")
print(f"  - is_cash_out, is_transfer, is_payment, is_cash_in: one-hot flags")

✓ Created fraud pattern features:
  - amount_count: number of transactions with same amount
  - is_duplicate_amount: flag for duplicate amounts
  - is_cash_out, is_transfer, is_payment, is_cash_in: one-hot flags


In [26]:
# FEATURE ENGINEERING - RATIO AND INTERACTION FEATURES
# Transaction type frequency
type_freq = df_clean.groupby('type')['step'].transform('count')
df_clean['type_frequency'] = type_freq / len(df_clean)

# Amount vs type average
type_amount_mean = df_clean.groupby('type')['amount'].transform('mean')
df_clean['amount_vs_type_avg'] = df_clean['amount'] / (type_amount_mean + 1)

# Time-based amount ratio
step_amount_mean = df_clean.groupby('step')['amount'].transform('mean')
df_clean['amount_vs_step_avg'] = df_clean['amount'] / (step_amount_mean + 1)

print(f"✓ Created ratio features:")
print(f"  - type_frequency: relative frequency of transaction type")
print(f"  - amount_vs_type_avg: amount ratio to type average")
print(f"  - amount_vs_step_avg: amount ratio to step average")

✓ Created ratio features:
  - type_frequency: relative frequency of transaction type
  - amount_vs_type_avg: amount ratio to type average
  - amount_vs_step_avg: amount ratio to step average


In [27]:
# FEATURE ENGINEERING - FLAGGED FRAUD ANALYSIS
# Interaction features with flagged fraud
df_clean['amount_flagged_interaction'] = df_clean['amount'] * df_clean['isFlaggedFraud']
df_clean['transfer_flagged'] = df_clean['is_transfer'] * df_clean['isFlaggedFraud']
df_clean['cashout_flagged'] = df_clean['is_cash_out'] * df_clean['isFlaggedFraud']

print(f"✓ Created interaction features with isFlaggedFraud:")
print(f"  - amount_flagged_interaction: amount * isFlaggedFraud")
print(f"  - transfer_flagged: TRANSFER * isFlaggedFraud")
print(f"  - cashout_flagged: CASH_OUT * isFlaggedFraud")


✓ Created interaction features with isFlaggedFraud:
  - amount_flagged_interaction: amount * isFlaggedFraud
  - transfer_flagged: TRANSFER * isFlaggedFraud
  - cashout_flagged: CASH_OUT * isFlaggedFraud


In [28]:
# SELECT FINAL FEATURE SET
# Feature selection based on correlation analysis and domain knowledge
feature_columns = [
    # Core features
    'step',
    'amount',
    'type_encoded',
    
    # Time-based features
    'hour_of_day',
    'day_of_week',
    'week_of_month',
    
    # Amount transformations
    'log_amount',
    'sqrt_amount',
    'zscore_amount',
    'amount_vs_type_avg',
    'amount_vs_step_avg',
    
    # Categorical flags
    'is_high_value',
    'is_very_high_value',
    
    # Fraud pattern features
    'amount_count',
    'is_duplicate_amount',
    'is_cash_out',
    'is_transfer',
    'is_payment',
    'is_cash_in',
    
    # Pattern features
    'type_frequency',
    'amount_flagged_interaction',
    'transfer_flagged',
    'cashout_flagged',
    
    # Flagged indicator
    'isFlaggedFraud'
]

X = df_clean[feature_columns]
y = df_clean['isFraud']

print(f"✓ Final feature set:")
print(f"  - Number of features: {len(feature_columns)}")
print(f"  - Features: {feature_columns}")
print(f"  - Target distribution:")
print(f"    - Fraud (1): {y.sum():,} ({y.sum()/len(y)*100:.4f}%)")
print(f"    - Non-Fraud (0): {len(y) - y.sum():,} ({100 - y.sum()/len(y)*100:.4f}%)")

✓ Final feature set:
  - Number of features: 24
  - Features: ['step', 'amount', 'type_encoded', 'hour_of_day', 'day_of_week', 'week_of_month', 'log_amount', 'sqrt_amount', 'zscore_amount', 'amount_vs_type_avg', 'amount_vs_step_avg', 'is_high_value', 'is_very_high_value', 'amount_count', 'is_duplicate_amount', 'is_cash_out', 'is_transfer', 'is_payment', 'is_cash_in', 'type_frequency', 'amount_flagged_interaction', 'transfer_flagged', 'cashout_flagged', 'isFlaggedFraud']
  - Target distribution:
    - Fraud (1): 8,213 (0.1291%)
    - Non-Fraud (0): 6,354,407 (99.8709%)


In [30]:
# Save feature list for reference
os.makedirs('./reports', exist_ok=True)
with open('./reports/feature_list.txt', 'w') as f:
    for feat in feature_columns:
        f.write(f"{feat}\n")

print(f"✓ Feature list saved to ./reports/feature_list.txt")

✓ Feature list saved to ./reports/feature_list.txt


In [31]:
# HANDLE CLASS IMBALANCE - STRATIFIED SPLIT
# Stratified train-test split to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"✓ Stratified split completed:")
print(f"  - Training set: {len(X_train):,} samples")
print(f"  - Test set: {len(X_test):,} samples")
print(f"  - Train fraud ratio: {y_train.sum()/len(y_train)*100:.4f}%")
print(f"  - Test fraud ratio: {y_test.sum()/len(y_test)*100:.4f}%")

✓ Stratified split completed:
  - Training set: 5,090,096 samples
  - Test set: 1,272,524 samples
  - Train fraud ratio: 0.1291%
  - Test fraud ratio: 0.1291%


In [32]:
# DATA SCALING
# Identify numerical features for scaling
numeric_features = ['step', 'amount', 'log_amount', 'sqrt_amount', 'zscore_amount',
                    'hour_of_day', 'day_of_week', 'week_of_month', 
                    'amount_vs_type_avg', 'amount_vs_step_avg',
                    'amount_count', 'type_frequency']

# Create original versions (for tree-based models)
X_train_orig = X_train.copy()
X_test_orig = X_test.copy()

# Create scaled versions (for linear and neural models)
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test[numeric_features])

print(f"✓ Scaling completed:")
print(f"  - Scaler used: StandardScaler")
print(f"  - Features scaled: {len(numeric_features)}")
print(f"  - Training set scaled: {X_train_scaled.shape}")
print(f"  - Test set scaled: {X_test_scaled.shape}")

✓ Scaling completed:
  - Scaler used: StandardScaler
  - Features scaled: 12
  - Training set scaled: (5090096, 24)
  - Test set scaled: (1272524, 24)


In [33]:
# Alternative scaling using RobustScaler (less sensitive to outliers)
robust_scaler = RobustScaler()
X_train_robust = X_train.copy()
X_test_robust = X_test.copy()
X_train_robust[numeric_features] = robust_scaler.fit_transform(X_train[numeric_features])
X_test_robust[numeric_features] = robust_scaler.transform(X_test[numeric_features])

print(f"✓ RobustScaler version also created (for comparison)")

✓ RobustScaler version also created (for comparison)


In [34]:
# SAVING PREPROCESSED DATA
# Create processed directory if not exists
os.makedirs('./data/processed', exist_ok=True)

# Save original (unscaled) versions
X_train_orig.to_csv('./data/processed/X_train.csv', index=False)
X_test_orig.to_csv('./data/processed/X_test.csv', index=False)

# Save scaled versions
X_train_scaled.to_csv('./data/processed/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('./data/processed/X_test_scaled.csv', index=False)

# Save robust scaled versions
X_train_robust.to_csv('./data/processed/X_train_robust.csv', index=False)
X_test_robust.to_csv('./data/processed/X_test_robust.csv', index=False)

# Save target variables
pd.Series(y_train).to_csv('./data/processed/y_train.csv', index=False)
pd.Series(y_test).to_csv('./data/processed/y_test.csv', index=False)

# Save scaler for deployment
os.makedirs('./models_saved', exist_ok=True)
joblib.dump(scaler, './models_saved/scaler.pkl')
joblib.dump(robust_scaler, './models_saved/robust_scaler.pkl')

print(f"✓ All processed data saved to ./data/processed/")
print(f"✓ Scalers saved to ./models_saved/")

✓ All processed data saved to ./data/processed/
✓ Scalers saved to ./models_saved/


In [39]:
def objective_random_forest(trial):

    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500, step=50),
        'max_depth': trial.suggest_int('max_depth', 5, 50, step=5),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical(
            'max_features',
            ['sqrt', 'log2', None]
        ),
        'class_weight': trial.suggest_categorical(
            'class_weight',
            ['balanced', 'balanced_subsample']
        ),

        # CPU
        'n_jobs': -1,
        'random_state': 42
    }

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = RandomForestClassifier(**params)

        model.fit(X_tr, y_tr)

        pred = model.predict(X_val)

        scores.append(f1_score(y_val, pred))

    return np.mean(scores)

def objective_xgboost(trial):

    params = {

        'n_estimators': trial.suggest_int(
            'n_estimators',
            100,
            500,
            step=50
        ),

        'max_depth': trial.suggest_int(
            'max_depth',
            3,
            12
        ),

        'learning_rate': trial.suggest_float(
            'learning_rate',
            0.01,
            0.3,
            log=True
        ),

        'subsample': trial.suggest_float(
            'subsample',
            0.6,
            1.0
        ),

        'colsample_bytree': trial.suggest_float(
            'colsample_bytree',
            0.6,
            1.0
        ),

        'min_child_weight': trial.suggest_int(
            'min_child_weight',
            1,
            7
        ),

        'gamma': trial.suggest_float(
            'gamma',
            0,
            0.5
        ),

        'scale_pos_weight': trial.suggest_float(
            'scale_pos_weight',
            5,
            50
        ),

        'tree_method': 'hist',
        'device': 'cuda',


        'eval_metric': 'logloss',

        'random_state': 42,

        'n_jobs': -1
    }

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = XGBClassifier(**params)

        model.fit(X_tr, y_tr)

        pred = model.predict(X_val)

        scores.append(f1_score(y_val, pred))

    return np.mean(scores)


def objective_lightgbm(trial):

    params = {

        'n_estimators': trial.suggest_int(
            'n_estimators',
            100,
            500,
            step=50
        ),

        'learning_rate': trial.suggest_float(
            'learning_rate',
            0.01,
            0.3,
            log=True
        ),

        'num_leaves': trial.suggest_int(
            'num_leaves',
            20,
            100
        ),

        'max_depth': trial.suggest_int(
            'max_depth',
            3,
            12
        ),

        'min_child_samples': trial.suggest_int(
            'min_child_samples',
            10,
            50
        ),

        'subsample': trial.suggest_float(
            'subsample',
            0.6,
            1.0
        ),

        'colsample_bytree': trial.suggest_float(
            'colsample_bytree',
            0.6,
            1.0
        ),

        'reg_alpha': trial.suggest_float(
            'reg_alpha',
            0,
            1
        ),

        'reg_lambda': trial.suggest_float(
            'reg_lambda',
            0,
            1
        ),

        'class_weight': trial.suggest_categorical(
            'class_weight',
            ['balanced', None]
        ),

        # CPU
        'n_jobs': -1
    }

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = LGBMClassifier(
            **params,
            random_state=42,
            verbose=-1
        )

        model.fit(X_tr, y_tr)

        pred = model.predict(X_val)

        scores.append(f1_score(y_val, pred))

    return np.mean(scores)



In [40]:
rf_study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=MedianPruner()
)

rf_study.optimize(
    objective_random_forest,
    n_trials=15,
    show_progress_bar=True
)

print(f"\n✓ Random Forest optimization completed!")
print(f"  - Best F1 Score: {rf_study.best_value:.4f}")
print(f"  - Best parameters:")
for key, value in rf_study.best_params.items():
    print(f"    - {key}: {value}")

print("\n🔍 Starting XGBoost optimization...")
print("-" * 60)

xgb_study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=MedianPruner()
)

xgb_study.optimize(
    objective_xgboost,
    n_trials=15,
    show_progress_bar=True
)

print(f"\n✓ XGBoost optimization completed!")
print(f"  - Best F1 Score: {xgb_study.best_value:.4f}")
print(f"  - Best parameters:")
for key, value in xgb_study.best_params.items():
    print(f"    - {key}: {value}")

[I 2026-07-21 00:24:53,245] A new study created in memory with name: no-name-75cc5fb6-8c93-4716-9ee0-ef8600200e6d


  0%|          | 0/15 [00:00<?, ?it/s]

[I 2026-07-21 00:33:41,295] Trial 0 finished with value: 0.48817321025951216 and parameters: {'n_estimators': 200, 'max_depth': 50, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 'sqrt', 'class_weight': 'balanced'}. Best is trial 0 with value: 0.48817321025951216.
[I 2026-07-21 00:54:05,491] Trial 1 finished with value: 0.04504031805564875 and parameters: {'n_estimators': 400, 'max_depth': 5, 'min_samples_split': 20, 'min_samples_leaf': 9, 'max_features': 'sqrt', 'class_weight': 'balanced_subsample'}. Best is trial 0 with value: 0.48817321025951216.
[I 2026-07-21 01:20:41,569] Trial 2 finished with value: 0.09169615755882082 and parameters: {'n_estimators': 250, 'max_depth': 15, 'min_samples_split': 13, 'min_samples_leaf': 2, 'max_features': None, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.48817321025951216.
[W 2026-07-21 01:52:22,095] Trial 3 failed with parameters: {'n_estimators': 300, 'max_depth': 30, 'min_samples_split': 2, 'min_samples_leaf': 7, '

KeyboardInterrupt: 

In [ ]:
# OPTUNA VISUALIZATION
# Create optimization visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Random Forest Optimization History
ax1 = axes[0, 0]
rf_trials = rf_study.trials_dataframe()
if len(rf_trials) > 0:
    ax1.plot(rf_trials['value'], 'b-', linewidth=2)
    ax1.axhline(y=rf_study.best_value, color='r', linestyle='--', label=f'Best: {rf_study.best_value:.4f}')
ax1.set_xlabel('Trial Number')
ax1.set_ylabel('F1 Score')
ax1.set_title('Random Forest: Optimization History')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Random Forest Parameter Importance
ax2 = axes[0, 1]
importance = optuna.importance.get_param_importances(rf_study)
top_params = dict(sorted(importance.items(), key=lambda x: x[1], reverse=True)[:5])
ax2.barh(list(top_params.keys()), list(top_params.values()), color='skyblue')
ax2.set_xlabel('Importance')
ax2.set_title('Random Forest: Top 5 Important Parameters')

# XGBoost Optimization History
ax3 = axes[1, 0]
xgb_trials = xgb_study.trials_dataframe()
if len(xgb_trials) > 0:
    ax3.plot(xgb_trials['value'], 'r-', linewidth=2)
    ax3.axhline(y=xgb_study.best_value, color='g', linestyle='--', label=f'Best: {xgb_study.best_value:.4f}')
ax3.set_xlabel('Trial Number')
ax3.set_ylabel('F1 Score')
ax3.set_title('XGBoost: Optimization History')
ax3.legend()
ax3.grid(True, alpha=0.3)

# XGBoost Parameter Importance
ax4 = axes[1, 1]
importance_xgb = optuna.importance.get_param_importances(xgb_study)
top_params_xgb = dict(sorted(importance_xgb.items(), key=lambda x: x[1], reverse=True)[:5])
ax4.barh(list(top_params_xgb.keys()), list(top_params_xgb.values()), color='lightcoral')
ax4.set_xlabel('Importance')
ax4.set_title('XGBoost: Top 5 Important Parameters')

plt.tight_layout()
os.makedirs('../reports/figures', exist_ok=True)
plt.savefig('../reports/figures/optuna_optimization.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# SAVE BEST PARAMETERS FOR LATER USE
# Store best parameters
best_params = {
    'random_forest': rf_study.best_params,
    'xgboost': xgb_study.best_params,
    'random_forest_f1': rf_study.best_value,
    'xgboost_f1': xgb_study.best_value
}

# Save to file
os.makedirs('./config', exist_ok=True)
with open('./config/best_params.json', 'w') as f:
    json.dump(best_params, f, indent=4)

print("✓ Best parameters saved to ./config/best_params.json")

In [ ]:
# GENERATING SUMMARY REPORT
DATE: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
DATASET: PaySim Mobile Money Transactions

FEATURES SUMMARY:
- Total features: {len(feature_columns)}
- Features used: {', '.join(feature_columns)}
- Training samples: {len(X_train):,}
- Test samples: {len(X_test):,}

CLASS DISTRIBUTION:
- Fraud: {y.sum():,} ({y.sum()/len(y)*100:.4f}%)
- Non-Fraud: {len(y) - y.sum():,} ({100 - y.sum()/len(y)*100:.4f}%)

OPTIMAL HYPERPARAMETERS:

RANDOM FOREST:
- Best F1 Score: {best_params['random_forest_f1']:.4f}
- Best parameters:
  {json.dumps(best_params['random_forest'], indent=4)}

XGBOOST:
- Best F1 Score: {best_params['xgboost_f1']:.4f}
- Best parameters:
  {json.dumps(best_params['xgboost'], indent=4)}
  
# Save summary
os.makedirs('./reports', exist_ok=True)
with open('./reports/feature_engineering_summary.txt', 'w') as f:
    f.write(summary)

print(f"✓ Summary report saved to ./reports/feature_engineering_summary.txt")

In [ ]:
print("\n📋 Summary of generated artifacts:")
print("  ✓ Processed data files (6 files)")
print("  ✓ Scaler models (2 files)")
print("  ✓ Feature list")
print("  ✓ Optimized hyperparameters (2 models)")
print("  ✓ Optimization visualizations")
print("  ✓ Summary report")

print("\n📁 Output files:")
print("  - ../data/processed/X_train.csv")
print("  - ../data/processed/X_test.csv")
print("  - ../data/processed/X_train_scaled.csv")
print("  - ../data/processed/X_test_scaled.csv")
print("  - ../data/processed/X_train_robust.csv")
print("  - ../data/processed/X_test_robust.csv")
print("  - ../data/processed/y_train.csv")
print("  - ../data/processed/y_test.csv")
print("  - ../models_saved/scaler.pkl")
print("  - ../models_saved/robust_scaler.pkl")
print("  - ../config/best_params.json")
print("  - ../reports/feature_engineering_summary.txt")
print("  - ../reports/figures/optuna_optimization.png")

print("\n🚀 Ready for Notebook 03: Model Training")